<div dir="rtl">

فایل نوت بوک NLP_02_DeBERTa_final توی محیط گوگل کولب انجام شده بود. چون حجم رایگان استفاده از GPUم تموم شد :) مجبور شدم از محیط کگل استفاده کنم
</div>

In [1]:
import os

print("Datasets available in Kaggle:\n")

for name in os.listdir("/kaggle/input"):
    print(name)
    path = os.path.join("/kaggle/input", name)

    if os.path.isdir(path):
        for root, dirs, files in os.walk(path):
            level = root.replace(path, "").count(os.sep)
            if level > 2:
                dirs[:] = []
                continue

            print("  " * (level + 1) + os.path.basename(root) + "/")

            for file in files:
                print("  " * (level + 2) + file)

Datasets available in Kaggle:

datasets
  datasets/
    kianaghaznavi/
      nlp-03-nlptown-final-dataset/
        test.csv
        test_cleaned.csv
        train.csv
        val.csv
      nlp-02-deberta/
        config.json
        model.safetensors
        tokenizer.json
        tokenizer_config.json


In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not enabled!")

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [3]:
import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [4]:
MODEL_PATH = "/kaggle/input/datasets/kianaghaznavi/nlp-02-deberta"

TEST_PATH = "/kaggle/input/datasets/kianaghaznavi/nlp-03-nlptown-final-dataset/test_cleaned.csv"

print("Model files:")
print(os.listdir(MODEL_PATH))

Model files:
['config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json']


In [5]:
test_df = pd.read_csv(TEST_PATH)

print("Shape:", test_df.shape)
print("Columns:", test_df.columns.tolist())

display(test_df.head(2))

Shape: (20000, 13)
Columns: ['vote', 'verified', 'reviewTime', 'reviewerID', 'asin', 'style', 'reviewerName', 'reviewText', 'summary', 'unixReviewTime', 'review_year', 'reviewText_clean', 'full_text']


,vote,verified,reviewTime,reviewerID,asin,style,reviewerName,reviewText,summary,unixReviewTime,review_year,reviewText_clean,full_text
0,0,1,2018-06-04,A20GGWE66JW9X2,B006Z394GM,{'Color:': ' FPS01-C'},Brian C Toner,The name and description of this device are mi...,The prize for most useless invention of all ti...,1528070400,2018,The name and description of this device are mi...,The prize for most useless invention of all ti...
1,0,1,2018-02-18,ARYJWXHEYHG9M,B005BE058W,"{'Size:': ' 1000W', 'Style:': ' G2'}",Snake,One of the molex connectors on the power suppl...,MELTED MOLEX CONNECTOR,1518912000,2018,One of the molex connectors on the power suppl...,MELTED MOLEX CONNECTOR One of the molex connec...


In [6]:
tok = AutoTokenizer.from_pretrained(MODEL_PATH)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH
)

model.to(device)
model.eval()

print("Model loaded successfully!")
print("Device:", device)
print("Number of classes:", model.config.num_labels)
print("Class mapping:", model.config.id2label)
print("Tokenizer max length:", tok.model_max_length)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Model loaded successfully!
Device: cuda
Number of classes: 5
Class mapping: {0: 'LABEL_0', 1: 'LABEL_1', 2: 'LABEL_2', 3: 'LABEL_3', 4: 'LABEL_4'}
Tokenizer max length: 1000000000000000019884624838656


In [7]:
print("Model config:")
print("Model type:", model.config.model_type)
print("Num labels:", model.config.num_labels)
print("Max position embeddings:", model.config.max_position_embeddings)

print("\nTokenizer:")
print("Tokenizer:", tok.name_or_path)
print("Padding side:", tok.padding_side)
print("Truncation side:", tok.truncation_side)

Model config:
Model type: deberta-v2
Num labels: 5
Max position embeddings: 512

Tokenizer:
Tokenizer: /kaggle/input/datasets/kianaghaznavi/nlp-02-deberta
Padding side: right
Truncation side: right


In [8]:
texts = (
    test_df["full_text"]
    .astype(str)
    .tolist()
)

print("Number of texts:", len(texts))
print("First text:", texts[0][:300])

Number of texts: 20000
First text: The prize for most useless invention of all time goes to: FPS01-C ... Conversion Tray for Laptop (Spoiler: its not for a laptop) The name and description of this device are misleading. It is supposed to be a conversion tray for a laptop optical drive. I assumed that you would be able to remove the o


<div dir="rtl">
برای اینکه پردازش سریع‌تر بشه، متن‌ها رو بر اساس طول مرتب می‌کنیم و بعد از prediction ترتیب اصلی رو برمی‌گردونیم.
</div>

In [9]:
def predict_probs_deberta(texts, batch_size=128, max_length=128):

    model.eval()

    # Sort texts by length
    sorted_indices = sorted(
        range(len(texts)),
        key=lambda i: len(texts[i])
    )

    sorted_texts = [texts[i] for i in sorted_indices]

    all_probs = []

    for start in tqdm(range(0, len(sorted_texts), batch_size)):

        batch_texts = sorted_texts[start:start + batch_size]

        inputs = tok(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        with torch.no_grad():

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):
                outputs = model(**inputs)
                probs = torch.softmax(outputs.logits, dim=1)

        all_probs.append(probs.float().cpu().numpy())

    # Concatenate predictions
    sorted_probs = np.concatenate(all_probs, axis=0)

    # Restore original row order
    final_probs = np.empty_like(sorted_probs)
    final_probs[sorted_indices] = sorted_probs

    return final_probs

In [10]:
deberta_final_probs = predict_probs_deberta(
    texts,
    batch_size=128,
    max_length=128
)

print("Prediction shape:", deberta_final_probs.shape)
print("First prediction:", deberta_final_probs[0])
print("Probability sum:", deberta_final_probs[0].sum())

  0%|          | 0/157 [00:00<?, ?it/s]

Prediction shape: (20000, 5)
First prediction: [0.8411429  0.10600354 0.03318492 0.01023263 0.00943595]
Probability sum: 0.99999994


In [11]:
np.save(
    "/kaggle/working/deberta_final_probs.npy",
    deberta_final_probs
)

print("DeBERTa predictions saved successfully!")

DeBERTa predictions saved successfully!
